# safecmd: Safe Command Execution

This guide explains Safe Mode in Dialeng, powered by safecmd - a tool for validating shell commands against an allowlist before execution.

## Overview

safecmd provides security for shell command execution by:
- Parsing bash commands into an Abstract Syntax Tree (AST)
- Validating all commands against a configurable allowlist
- Blocking dangerous operations like `rm -rf`, `sudo`, etc.
- Preventing command injection attacks

## When to Use Safe Mode

Enable Safe Mode when:

1. **LLM-generated commands** - When an AI assistant generates shell commands
2. **Shared notebooks** - When sharing notebook URLs with others
3. **Untrusted input** - When commands come from external sources
4. **Learning environments** - To prevent accidental destructive operations

## Prerequisites

safecmd requires the `shfmt` binary for bash parsing:

```bash
# macOS
brew install shfmt

# Ubuntu/Debian
sudo apt install shfmt

# Or download from GitHub
# https://github.com/mvdan/sh/releases
```

**Note:** If shfmt is not installed, Safe Mode will be disabled but shell execution will still work.

## Enabling Safe Mode

Safe Mode is a **notebook-level setting**. To enable it:

1. Look for the **Safe** checkbox in the notebook toolbar (next to the model selector)
2. Check the box to enable Safe Mode
3. All shell commands in the notebook will now be validated

When Safe Mode is enabled:
- A green "Safe" badge appears in shell cell headers
- Commands are validated before execution
- Disallowed commands raise a `DisallowedCmd` error

## Default Allowlist

safecmd includes a generous allowlist of read-only and easily-reverted commands:

### File Viewing
- `cat`, `head`, `tail`, `less`, `more`, `bat`

### Directory Operations
- `ls`, `tree`, `locate`, `pwd`, `cd`

### Text Search
- `grep`, `rg`, `ag`, `ack`, `fgrep`, `egrep`

### Text Processing
- `cut`, `sort`, `uniq`, `wc`, `tr`, `column`

### File Analysis
- `file`, `stat`, `du`, `df`, `which`, `whereis`, `type`

### Git (Read-only)
- `git log`, `git show`, `git diff`, `git status`
- `git branch`, `git tag`, `git remote`
- `git blame`, `git shortlog`, `git describe`

### Git (Workspace)
- `git fetch`, `git add`, `git commit`
- `git switch`, `git checkout`

### Network (Read-only)
- `curl`, `wget` (for fetching only)
- `ping`, `dig`, `nslookup`, `host`

### System Info
- `date`, `cal`, `uptime`, `whoami`
- `hostname`, `uname`, `env`, `printenv`

### Utilities
- `echo`, `printf`, `yes`, `seq`
- `basename`, `dirname`, `realpath`

## Blocked Operations

The following are **always blocked** in Safe Mode:

### Destructive Commands
- `rm`, `rmdir`, `unlink` - File deletion
- `mv` (to restricted destinations) - Moving files
- `chmod`, `chown` - Permission changes

### Privilege Escalation
- `sudo`, `su`, `doas` - Admin access
- `pkexec` - PolicyKit execution

### System Modification
- `shutdown`, `reboot`, `halt` - System control
- `kill`, `killall` - Process termination
- `dd` - Raw disk operations

### find Restrictions
- `find -exec`, `-execdir` - Arbitrary execution
- `find -delete` - Deletion via find
- `find -ok`, `-okdir` - Interactive execution

## Examples

### Allowed Commands

These commands work in Safe Mode (enable Safe Mode in toolbar to test):

In [ ]:
# List files (use ! or %%bash for commands with flags)
!ls -la

In [ ]:
%%bash
# Search in files
grep -r "def " *.py 2>/dev/null | head -5 || echo "No matches"

In [ ]:
# Git operations - simple command works with %bash
%bash git status

### Blocked Commands

With Safe Mode enabled, these commands will be blocked:

In [ ]:
# These commands are BLOCKED in Safe Mode
# Uncomment to test (will raise DisallowedCmd)

# %%bash
# rm -rf /tmp/test  # File deletion

# !sudo ls          # Privilege escalation

# %%bash
# echo "test" > /etc/test  # Write to system path

## How It Works

safecmd uses a multi-stage validation process:

### 1. AST Parsing
Commands are parsed into an Abstract Syntax Tree using `shfmt`. This correctly handles:
- Pipelines (`cmd1 | cmd2`)
- Command substitutions (`$(cmd)`)
- Subshells (`(cmd)`)
- Redirections (`> file`)

### 2. Command Extraction
All executable commands are extracted from the AST, including nested commands.

### 3. Allowlist Validation
Each command is checked against the allowlist using prefix matching.

### 4. Operator Validation
Only permitted operators are allowed:
- `|` (pipe)
- `<` (input redirection)
- `&&` (and)
- `||` (or)
- `;` (sequential)

**Blocked operators:**
- `>` (output redirection) - Can overwrite files
- `>>` (append) - Can modify files

## Security Considerations

### What Safe Mode Protects Against

1. **Accidental destruction** - `rm -rf /` won't run
2. **Command injection** - Nested malicious commands are detected
3. **Privilege escalation** - `sudo` and similar are blocked
4. **File system writes** - Output redirection is restricted

### What Safe Mode Does NOT Protect Against

1. **Deliberate bypass attempts** - A determined attacker may find workarounds
2. **Network exfiltration** - `curl` can still send data
3. **Resource exhaustion** - Fork bombs, infinite loops
4. **Complete sandboxing** - It's command validation, not containerization

**Safe Mode is a defense layer, not a complete security solution.** For untrusted code execution, consider additional measures like containers or VMs.

## Customizing the Allowlist

safecmd allows customizing the allowlist via configuration files:

```ini
# ~/.config/safecmd/config.ini (Linux)
# ~/Library/Application Support/safecmd/config.ini (macOS)

[commands]
my_custom_tool = true
my_other_tool = true

[destinations]
./my_project = true
/tmp = true
```

See the [safecmd documentation](https://github.com/AnswerDotAI/safecmd) for details.

## Next Steps

- See `pshnb_guide.ipynb` for general shell usage
- See `shell_integration.ipynb` for the complete integration overview
- Enable Safe Mode in this notebook to test validation!